# Scalar Field Analysis Engine - Quickstart Notebook

Symbolic, numeric, and visualization helpers for two-variable scalar fields.

This notebook is the working surface for the package. It lets you define a scalar field `f(x, y)`, differentiate it symbolically, evaluate it numerically, inspect its structure, and open a GPU-backed 3D surface viewer.

The package includes:

- symbolic differentiation with SymPy
- grid evaluation with NumPy
- critical-point candidates from zero-contour intersections
- Matplotlib summary plots
- a GPU-backed VisPy 3D surface viewer

---

## 3D Viewer

The notebook-safe viewer entrypoint is:

```python
from scalar_field_analysis.rendering import launch_3d_viewer

viewer_process = launch_3d_viewer(result)
```

For direct script usage:

```python
from scalar_field_analysis.rendering import show_surface_3d

show_surface_3d(result)
```

The legacy notebook import remains supported:

```python
from Renderer.bridge import launch_viewer_subprocess

viewer_process = launch_viewer_subprocess(result, backend="pyqt6")
```

---

## Environment & Dependencies

This project relies on a small, explicit stack:

| Package | Purpose |
| --- | --- |
| `sympy` | Symbolic mathematics |
| `numpy` | Numerical evaluation |
| `scipy` | Distance computations for contour intersections |
| `matplotlib` | 2D visualization |
| `vispy` | GPU-backed 3D rendering |
| `PyQt6` | Native VisPy window backend |

## Tested Baseline

```text
python==3.14.3
sympy==1.14.0
numpy==2.4.4
scipy==1.17.1
matplotlib==3.10.8
vispy==0.16.1
PyQt6==6.11.0
```

These version numbers describe the local tested environment. The package metadata in `pyproject.toml` is the source of truth for install requirements.

---

## Environment Setup

Use the project virtual environment selected in VS Code: `.venv (3.14.3)`. If setting up from scratch, create and activate a virtual environment, then install the package in editable mode:

```powershell
py -3.14 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -e ".[render3d,test]"
python -m ipykernel install --user --name scalar-field-venv --display-name "Python (.venv)"
```

---

## What This Notebook Does

1. Define a scalar field:

    ```python
    f = x**3 - 3*x*y**2
    ```

2. Run it through the analysis pipeline.

3. Visualize:

   - the scalar field
   - its partial derivatives
   - zero-contours of the gradient
   - approximate critical points
   - the GPU-backed 3D surface

# Jupyter Kernel Troubleshooting (VS Code)

If you encounter errors like:

```text
ModuleNotFoundError: No module named 'sympy'
```

even after installing dependencies, the notebook is likely not using the project virtual environment.

## Fix

1. Activate the virtual environment:

    ```powershell
    .\.venv\Scripts\Activate.ps1
    ```

2. Register it as a Jupyter kernel:

    ```powershell
    python -m ipykernel install --user --name scalar-field-venv --display-name "Python (.venv)"
    ```

3. Restart the kernel, or restart VS Code if necessary.

4. In the notebook kernel picker, select `.venv (3.14.3)` or `Python (.venv)`.

# Important Notes

* Critical points are **numerically approximated**, not solved exactly.
* Results depend on:

  * grid resolution
  * contour extraction
  * intersection thresholds
* Flat regions and near-degenerate gradients may produce noise.

This is an *analytical probe*, not a theorem prover.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

In [ ]:
from field_analysis import GridSpec, analyse_scalar_field
from pixel_clamp import graph_is_oversampled, prepare_plot_result
from plotting import plot_analysis_summary

In [ ]:
# Symbolic variables
x, y = sp.symbols("x y", real=True)


# === 1. Define the scalar field ===
# Replace this expression with any reasonable scalar field f(x, y).

expr = 1/32 * (3 * x**2 + 3 * y**2 + 30 * x - 20 * y + 325)


# === 2. Configure the analysis grid ===
# Notes:
# - Larger resolutions give cleaner contours, but cost more time.
# - Wider ranges reveal more global structure, but may hide local detail.
x_range=(-7.5, -2.5)
y_range=(0, 5)

resolution=(400, 400)
ANTI_OVERSAMPLE = True  # Set to False to disable anti-oversampling
                        # (not recommended)

grid = GridSpec(
    x_range, y_range,
    resolution,
)


In [ ]:
# === 3. Run the scalar-field analysis ===
result = analyse_scalar_field(
    expr,
    variables=(x, y),
    grid=grid,
    intersection_threshold=5e-2,
    deduplication_tolerance=7.5e-2,
)


# === 4. Inspect the symbolic outputs ===
print("Scalar field:")
print("f(x, y) =", result.expr)
print()

print("Gradient:")
print("df/dx =", result.dfdx)
print("df/dy =", result.dfdy)
print()

print("Approximate critical points:")
if result.critical_points.size == 0:
    print("None detected on the current grid.")
else:
    for i, point in enumerate(result.critical_points, start=1):
        px, py = point
        print(f"{i}. ({px:.6f}, {py:.6f})")

In [ ]:
# === 5. Plot a compact summary view ===
if ANTI_OVERSAMPLE and graph_is_oversampled(result,
                                            target_display_pixels=(1280, 720),
                                           ):
    print("Plotting grid exceeds 720p-equivalent display density for a 2x2 summary view.")
    print("Downsampling for display only...")
    clamped_result = prepare_plot_result(result)
else:  # No oversampling detected, or anti-oversampling disabled.
    clamped_result = result

fig, axes = plot_analysis_summary(
    clamped_result,
    show_filled_field=True,
    show_gradient=False,
)

plt.show()

In [ ]:
# === 6. Optional: inspect the Hessian at a candidate point ===
# This is not part of the main numerical pipeline, but it can be useful for
# local interpretation once a candidate has been found.


# === HESSIAN FORMATTING CONFIG ===
# Define column metadata: Name, Width, Justification, Precision, Etc.
COL_EXPR = {"name": "Symbolic Expression", "width": 20, "align": "<"}
COL_LABEL = {"name": "Derivative", "width": 10, "align": "<"}
COL_VALUE = {"name": "Numerical Value", "width": 15, "align": "^"}

COLUMNS = [COL_LABEL, COL_VALUE, COL_EXPR]
MARGIN_CHAR = "="
SEPARATION = "  "  # Spacing char between columns

# Unicode Box Drawing Characters
TOP_LEFT, TOP_RIGHT = "┌", "┐"
MID_LEFT, MID_RIGHT = "│", "│"
BOT_LEFT, BOT_RIGHT = "└", "┘"

MATRIX_VALUE_WIDTH = 8  # Slightly narrower for matrix display
PRECISION = ".2f"
JUSTIFICATION = "^"  # Change to '<', '^', or '>' as needed


# === FORMATTING ENGINE ===
def get_table_assets(columns):
    """Generates a header that accounts for inter-column spacing."""
    header_parts = []
    for c in columns:
        header_parts.append(f"{c['name']:{c['align']}{c['width']}}")

    header = SEPARATION.join(header_parts)
    return header


def format_hessian_rows(entries, subs, columns):
    """Yields rows where every cell is strictly justified and padded."""
    for label, expr in entries:
        val = float(sp.N(expr.subs(subs)))

        # Apply formatting requirements to each specific column
        #   - Label (Col 0),
        #   - Value (Col 1),
        #   - Expression (Col 2)
        c0 = f"{label:{columns[0]['align']}{columns[0]['width']}}"
        c1 = f"{val:{columns[1]['align']}{columns[1]['width']}.4f}"
        c2 = f"{str(expr):{columns[2]['align']}{columns[2]['width']}}"

        yield SEPARATION.join([c0, c1, c2])


def format_matrix_output(matrix, subs, width=MATRIX_VALUE_WIDTH, align=JUSTIFICATION):
    """Formats a SymPy matrix with mathematical brackets and alignment."""
    # 1. Evaluate to float grid
    data = matrix.subs(subs).tolist()

    # 2. Build the rows using dynamic formatting string
    fmt = f"{align}{width}{PRECISION}"
    formatted_rows = [[f"{float(val):{fmt}}" for val in r] for r in data]

    output = ["\nHessian Matrix Form:"]
    n_rows = len(formatted_rows)

    for i, row_data in enumerate(formatted_rows):
        content = " ".join(row_data)

        # 3. Select brackets based on row position
        if n_rows == 1:
            left, right = "[", "]"
        elif i == 0:
            left, right = TOP_LEFT, TOP_RIGHT
        elif i == n_rows - 1:
            left, right = BOT_LEFT, BOT_RIGHT
        else:
            left, right = MID_LEFT, MID_RIGHT

        output.append(f"{left} {content} {right}")

    return "\n".join(output)

# === EXECUTION (Logic Pipeline) ===
if result.critical_points.size != 0:
    for i, point in enumerate(result.critical_points, start=1):
        px, py = point
        substitutions = {x: float(px), y: float(py)}

        # Define the Hessian components
        hessian_data = [
            ("f_xx", sp.diff(result.dfdx, x)),
            ("f_xy", sp.diff(result.dfdx, y)),
            ("f_yx", sp.diff(result.dfdy, x)),
            ("f_yy", sp.diff(result.dfdy, y)),
        ]

        # Generate UI elements
        title = f"=== Hessian Analysis at Point {i}: ({px:.3f}, {py:.3f}) ==="
        header = get_table_assets(COLUMNS)
        margin = MARGIN_CHAR * len(title)

        print(f"\n{title}")
        print(header)
        print(margin)

        for row in format_hessian_rows(hessian_data, substitutions, COLUMNS):
            print(row)

        # Final Matrix Visualization
        H_mat = sp.Matrix(
            [
                [hessian_data[0][1], hessian_data[1][1]],
                [hessian_data[2][1], hessian_data[3][1]],
            ]
        )
        print(format_matrix_output(H_mat, substitutions)+"\n")

In [ ]:
# === 6. Launch 3D Viewer ===
from scalar_field_analysis.rendering import Surface3DConfig, launch_3d_viewer

viewer_process = launch_3d_viewer(
    result,
    config=Surface3DConfig(backend="pyqt6"),
)